In [ ]:
import pandas as pd
import numpy as np
import shapely.geometry as geom
from shapely.geometry import Point, Polygon
import anndata
import scanpy as sc

adata = sc.read_h5ad('path/to/xenium/combined_file.h5ad')
adata.obs_names_make_unique()

os.makedirs("marker_genes_per_sample", exist_ok=True)
os.makedirs("adata_sub_per_sample", exist_ok=True)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata)
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
sc.tl.leiden(adata, key_added='leiden', resolution= 0.3)  
sc.pl.spatial(
    adata,
    color="leiden",
    spot_size=15,
    img_key="hires",
    show=True, save = '0.3_leiden_clustering.pdf'
)
sc.tl.rank_genes_groups(adata, 'leiden', method='t-test')
markers_df = sc.get.rank_genes_groups_df(adata, group=None)
output_file = os.path.join(output_dir, "0.3_marker_genes_.csv")
markers_df.to_csv(output_file, index=False)
sc.pl.rank_genes_groups_dotplot(
        adata, 
        groupby="leiden", 
        standard_scale="var", 
        n_genes=5, 
        save='resolution_0.3.pdf',
    dendrogram = False
    )